# 05 — Chunk policy_documents into policy_chunks

This notebook turns the 30 full-document rows of
`${catalog}.pawshield.policy_documents` (produced by c0401) into the chunked,
metadata-rich `${catalog}.pawshield.policy_chunks` table that the index builder c0801
indexes in Vector Search.

The notebook walks two strategies and writes only one:

1. **Fixed-size** chunking (the easy default). The verified Section 4.7
   measurements show this fragments the canonical "right chunk" across
   multiple pieces at the chunk sizes that actually fit downstream
   retrieval budgets.
2. **Structure-aware** chunking. Split on section headings first, then
   fall back to a size limit only for sections longer than the limit.
   Section 4.7 stays whole regardless of size budget because the
   heading boundary aligns with the natural read unit.

The Delta target enables Change Data Feed (CDF), which is required for
the Delta Sync Vector Search index — without CDF, Vector Search
cannot incrementally sync only the rows that changed.

**Prerequisites**: the extraction notebook c0401 completed; `policy_documents` table
exists with 30 rows.

In [0]:
dbutils.widgets.text("catalog", "genaicert")
catalog = dbutils.widgets.get("catalog")
print(f"catalog: {catalog}")

catalog: genaicert


## 1. Confirm the source table

In [0]:
%sql
SELECT count(*) AS doc_count,
       sum(length(text)) AS total_chars
FROM IDENTIFIER(:catalog || '.pawshield.policy_documents');

doc_count,total_chars
30,3018613


## 2. Strategy A — fixed-size chunking (the contrast)

We size chunks at 1,500 characters with a 150-character overlap.
Section 4.7 is longer than a single 1,500-char window (the printed measurements table below has the exact figure), so fixed-window extraction splits it into two chunks —
roughly *preconditions* + *handling rules*. A reader who asks "do
chronic claims still hit the deductible?" needs both halves.

In [0]:
def chunk_fixed(text: str, size: int = 1500, overlap: int = 150):
    """Naive character-window chunker; no semantic boundary awareness."""
    chunks = []
    pos = 0
    n = len(text)
    while pos < n:
        chunks.append(text[pos: pos + size])
        pos += size - overlap
    return chunks

In [0]:
import re

# Spot-check on the canonical TX PawPlus doc.
sarah_doc = spark.sql(
    f"SELECT text FROM {catalog}.pawshield.policy_documents "
    "WHERE doc_id = 'pp_plus_tx_v3'"
).first().text

print(f"Document: pp_plus_tx_v3, {len(sarah_doc)} chars")
print(f"Fixed-size(1500/150) → {len(chunk_fixed(sarah_doc))} chunks")

# Locate where 4.7 lives to show its character span. (The authoritative
# per-size fragmentation table is computed below with find_section_47_range,
# which uses this same heading-delimiter regex.)
m = re.search(r"4\.7 Chronic condition exclusions", sarah_doc)
if m:
    section_start = m.start()
    next_heading = re.search(r"\n[0-9]+\.[0-9]+ ", sarah_doc[section_start + len("4.7 Chronic condition exclusions"):])
    section_end = section_start + len("4.7 Chronic condition exclusions") + (next_heading.start() if next_heading else len(sarah_doc))
    print(f"Section 4.7 spans chars [{section_start}, {section_end}) — {section_end - section_start} chars")

Document: pp_plus_tx_v3, 101104 chars
Fixed-size(1500/150) → 75 chunks
Section 4.7 spans chars [9919, 11671) — 1752 chars


## 3. Strategy B — structure-aware chunking (what we keep)

Every policy PDF in this corpus uses numbered headings (`1.1`, `4.7`,
`Section 5`). Splitting on those boundaries first, then sub-splitting
only the rare oversized section, gives chunks that align with how
the document was authored. Section 4.7 — the recurring "right chunk"
retrieved throughout the case study — stays whole at any practical size.

In [0]:
# Headings observed in the corpus: "N.N Title" or "Section N — Title".
HEADING_RE = re.compile(
    r"(?m)^\s*(?:(\d+\.\d+)\s+([^\n]+)|Section\s+(\d+)\s+[—-]\s+([^\n]+))"
)

# Sections longer than this get sub-split on paragraph boundaries.
MAX_CHUNK_CHARS = 2000


def chunk_structure_aware(text: str):
    """Yield (section_label, chunk_text) tuples for one document."""
    matches = list(HEADING_RE.finditer(text))
    if not matches:
        yield ("unlabelled", text.strip())
        return

    # Cover any text before the first heading.
    if matches[0].start() > 0:
        prefix = text[: matches[0].start()].strip()
        if prefix:
            yield ("front-matter", prefix)

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if not body:
            continue
        if m.group(1):
            label = f"{m.group(1)} {m.group(2).strip()}"
        else:
            label = f"Section {m.group(3)} {m.group(4).strip()}"

        if len(body) <= MAX_CHUNK_CHARS:
            yield (label, body)
            continue

        # Oversized section: split on blank lines, accumulate up to budget.
        parts = re.split(r"\n\s*\n", body)
        buf = ""
        for part in parts:
            if len(buf) + len(part) + 2 <= MAX_CHUNK_CHARS:
                buf = (buf + "\n\n" + part).strip() if buf else part
            else:
                if buf:
                    yield (label, buf)
                buf = part
        if buf:
            yield (label, buf)

In [0]:
# Verify Section 4.7 lands in a single chunk for the canonical doc.
labels = [(lab, len(txt)) for lab, txt in chunk_structure_aware(sarah_doc)]
sec_4_7 = [(lab, n) for lab, n in labels if lab.startswith("4.7")]
print(f"Structure-aware: {len(labels)} chunks total")
print(f"Section 4.7 chunks: {sec_4_7}")

Structure-aware: 38 chunks total
Section 4.7 chunks: [('4.7 Chronic condition exclusions and reimbursement handling', 1751)]


## 3b. Fragmentation comparison — fixed-size sweep + structure-aware

The table this cell prints compares four fixed-size budgets (500, 1000,
1500, 3000 chars, 10% overlap) against structure-aware on the
canonical `pp_plus_tx_v3` doc. The point is to show that *every*
fixed-size budget below the §4.7 section length fragments the
clause across multiple chunks, while structure-aware keeps it
whole regardless of size budget. The printed table below is the
source of truth for these fragmentation numbers.

In [0]:
import re

# Fixed-size budgets to sweep. 10% overlap is the standard heuristic
# (chunk_fixed defaults to size=1500/overlap=150, the 10% case).
FRAG_SIZES = [500, 1000, 1500, 3000]


def find_section_47_range(doc: str) -> tuple[int, int]:
    """Locate §4.7's character range [start, end) in the source doc.

    Searches for the §4.7 heading, then for the next section heading
    (regex `\\n[0-9]+\\.[0-9]+ `) to delimit the end. Falls back to
    `len(doc)` if no following section exists.
    """
    start_marker = "4.7 Chronic condition exclusions"
    start = doc.index(start_marker)
    next_heading = re.search(r"\n[0-9]+\.[0-9]+ ", doc[start + len(start_marker):])
    end = (start + len(start_marker) + next_heading.start()) if next_heading else len(doc)
    return (start, end)


def count_47_fragments(text_len: int, size: int, overlap: int,
                       sec_start: int, sec_end: int) -> int:
    """Count fixed-size chunks whose [pos, pos+size) overlaps §4.7's range.

    This is the fragmentation count — how many chunks the §4.7 content
    is split across (with overlap accounted for), not "chunks containing
    the heading string."
    """
    fragments = 0
    pos = 0
    while pos < text_len:
        chunk_end = min(pos + size, text_len)
        if pos < sec_end and chunk_end > sec_start:
            fragments += 1
        pos += size - overlap
    return fragments


def count_47_structure(structure_chunks_labeled) -> int:
    """Count structure-aware chunks whose label starts with '4.7'."""
    return sum(1 for lab, _txt in structure_chunks_labeled if lab.startswith("4.7"))


SEC_47_START, SEC_47_END = find_section_47_range(sarah_doc)

In [0]:
print(f"§4.7 range in pp_plus_tx_v3: chars [{SEC_47_START}, {SEC_47_END}) — "
      f"{SEC_47_END - SEC_47_START} chars")
print()

# Build rows for each fixed-size budget plus the structure-aware result.
rows = []
for size in FRAG_SIZES:
    rows.append((
        f"Fixed-size, {size} chars",
        len(chunk_fixed(sarah_doc, size=size, overlap=size // 10)),
        count_47_fragments(len(sarah_doc), size, size // 10, SEC_47_START, SEC_47_END),
    ))

# Structure-aware: count §4.7-labeled chunks (fragmentation under structure).
# Materialise the generator with list(...) so we can both take len() and
# iterate it in count_47_structure without exhausting it twice.
structure_chunks_labeled = list(chunk_structure_aware(sarah_doc))
rows.append((
    "Structure-aware (heading + 2000)",
    len(structure_chunks_labeled),
    count_47_structure(structure_chunks_labeled),
))

# Print a markdown-shaped table summarising the fragmentation sweep.
print(f"Fragmentation on pp_plus_tx_v3 ({len(sarah_doc)} chars)")
print()
print(f"| {'Strategy':<35} | {'Total chunks':>12} | {'§4.7 fragments':>14} |")
print(f"|{'-' * 37}|{'-' * 14}|{'-' * 16}|")
for label, total, frags in rows:
    print(f"| {label:<35} | {total:>12} | {frags:>14} |")

§4.7 range in pp_plus_tx_v3: chars [9919, 11671) — 1752 chars

Fragmentation on pp_plus_tx_v3 (101104 chars)

| Strategy                            | Total chunks | §4.7 fragments |
|-------------------------------------|--------------|----------------|
| Fixed-size, 500 chars               |          225 |              5 |
| Fixed-size, 1000 chars              |          113 |              3 |
| Fixed-size, 1500 chars              |           75 |              2 |
| Fixed-size, 3000 chars              |           38 |              2 |
| Structure-aware (heading + 2000)    |           38 |              1 |


## 4. Apply structure-aware chunking to all 30 documents

The chunker runs as a flatMap over the `policy_documents` rows,
carrying `doc_id`, `tier`, `state` to every produced chunk so
Vector Search can filter on them at retrieval time.

In [0]:
from pyspark.sql import Row


def doc_to_chunk_rows(doc):
    for idx, (section, body) in enumerate(chunk_structure_aware(doc.text)):
        yield Row(
            chunk_id=f"{doc.doc_id}__chunk_{idx:03d}",
            doc_id=doc.doc_id,
            tier=doc.tier,
            state=doc.state,
            section=section,
            chunk_text=body,
            # Token count: ~4 chars per token for English text.
            token_count=int(len(body) / 4),
        )

In [0]:
docs = spark.sql(
    f"SELECT doc_id, tier, state, text FROM {catalog}.pawshield.policy_documents"
).collect()

chunk_rows = [r for d in docs for r in doc_to_chunk_rows(d)]
print(f"Produced {len(chunk_rows)} chunks across {len(docs)} documents")

Produced 1140 chunks across 30 documents


In [0]:
chunks_df = spark.createDataFrame(chunk_rows)

In [0]:
display(chunks_df.limit(5))

chunk_id,doc_id,tier,state,section,chunk_text,token_count
pp_basic_ca_v3__chunk_000,pp_basic_ca_v3,WhiskerBasic,CA,front-matter,PawShield WhiskerBasic Policy contract — CA resident edition (Version 3) The WhiskerBasic plan provides accident-only coverage for unexpected injuries. It is the most affordable PawShield tier and is appropriate for Policyholders whose primary concern is catastrophic event protection. Illness-related claims are not reimbursed under WhiskerBasic. Section 1,90
pp_basic_ca_v3__chunk_001,pp_basic_ca_v3,WhiskerBasic,CA,1.1 Definitions,"1.1 Definitions For the purposes of this policy: ""Pet"" means the animal listed on the declarations page of this contract. ""Policyholder"" means the named adult listed as account-holder. ""Covered Veterinarian"" means any licensed veterinarian practicing in the United States, subject to the Network Status provisions of Section 2.4. ""Reimbursable Amount"" means the amount actually paid out by PawShield after application of the annual deductible and the per-claim copay percentage as specified in the declarations page.",130
pp_basic_ca_v3__chunk_002,pp_basic_ca_v3,WhiskerBasic,CA,1.2 Contract overview,"1.2 Contract overview This insurance contract is between the Policyholder and PawShield Insurance Inc. (""PawShield""). It governs the reimbursement of veterinary expenses incurred by the Policyholder for the care of the named Pet, subject to the terms, limits, and exclusions set forth herein. Coverage is limited to expenses incurred during the Policyholder's Policy Period as listed on the declarations page. Section 2",106
pp_basic_ca_v3__chunk_003,pp_basic_ca_v3,WhiskerBasic,CA,2.1 Covered services,"2.1 Covered services Subject to Section 4 (Exclusions) and the applicable deductible and copay, this policy reimburses the Policyholder for veterinary services rendered to the Pet that fall into the following categories: (a) accident-related diagnosis and treatment; (b) illness-related diagnosis and treatment; (c) prescription medications; (d) emergency and specialty care; and (e) for policies including the wellness rider, routine preventive care as enumerated in Section 2.3.",121
pp_basic_ca_v3__chunk_004,pp_basic_ca_v3,WhiskerBasic,CA,2.4 Network status,"2.4 Network status A Covered Veterinarian may be classified as either ""in-network"" or ""out-of-network"". In-network claims are reimbursed at the copay percentage specified on the declarations page. Out-of-network claims are reimbursed at five (5) percentage points below the in-network rate. A list of in-network veterinarians is maintained on the PawShield website and is updated quarterly. Section 3",101


## 5. Write to Delta with Change Data Feed enabled

CDF is required for the Delta Sync Vector Search index. Without
it, Vector Search cannot detect which rows changed and would need
to re-embed the entire table on every sync. Enabling CDF here is a
one-time, free-on-write cost.

In [0]:
target = f"{catalog}.pawshield.policy_chunks"
print(f"target: {target}")

target: genaicert.pawshield.policy_chunks


In [0]:
(
    chunks_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

# Source: https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search
# Delta Sync Vector Search indexes require Change Data Feed on the source
# table: "For standard endpoints, the source table must have Change Data
# Feed enabled." (CDF doc: https://docs.databricks.com/aws/en/delta/delta-change-data-feed)
spark.sql(f"""
    ALTER TABLE {target}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Wrote {chunks_df.count()} rows to {target} (CDF enabled)")

Wrote 1140 rows to genaicert.pawshield.policy_chunks (CDF enabled)


In [0]:
%sql
SELECT
  tier,
  state,
  count(*) AS chunk_count,
  round(avg(token_count)) AS avg_tokens,
  max(token_count)        AS max_tokens
FROM IDENTIFIER(:catalog || '.pawshield.policy_chunks')
GROUP BY tier, state
ORDER BY tier, state;

tier,state,chunk_count,avg_tokens,max_tokens
FureverPremier,CA,38,644.0,1521
FureverPremier,CO,38,665.0,1640
FureverPremier,FL,38,665.0,1640
FureverPremier,GA,38,663.0,1632
FureverPremier,IL,38,665.0,1587
FureverPremier,MA,38,668.0,1587
FureverPremier,NY,38,666.0,1602
FureverPremier,OR,38,654.0,1560
FureverPremier,TX,38,649.0,1567
FureverPremier,WA,38,649.0,1637


## 6. Verify the canonical right chunk is intact

In [0]:
%sql
SELECT chunk_id, section, length(chunk_text) AS chars, token_count
FROM IDENTIFIER(:catalog || '.pawshield.policy_chunks')
WHERE doc_id = 'pp_plus_tx_v3'
  AND section LIKE '4.7%';

chunk_id,section,chars,token_count
pp_plus_tx_v3__chunk_018,4.7 Chronic condition exclusions and reimbursement handling,1751,437


## What's next

The index builder c0801 creates a Vector Search Delta Sync index over this table with
`state` and `tier` as filterable metadata columns, using
`databricks-gte-large-en` as the managed embedding model.